# AKTU Autonomy Backend – Colab Runner

Run the **FastAPI backend** from **Google Colab**.

**Steps:**
1. **(Optional)** Run the **first code cell** to delete any existing database file for a fresh migration run. Skip if you want to keep existing data.
2. Run the **second code cell**. It will try to connect Google Drive (you may see a popup). If Drive fails with "credential propagation was unsuccessful", the notebook will use local storage instead so the backend still runs. When asked for **GitHub PAT**, paste your token (from https://github.com/settings/tokens, with **repo** scope).
3. Run the **third code cell** to start the server and get a public URL. You will be asked for your **ngrok authtoken** (free at https://dashboard.ngrok.com/signup → https://dashboard.ngrok.com/get-started/your-authtoken).
4. Keep this tab open while you use the API.

**If Drive mount fails:** You can ignore it. The backend will use Colab’s local disk; data will be lost when the session ends, but you’ll get a working API URL for this session.


# AKTU Autonomy Backend – Colab Runner\n\nThis notebook installs dependencies, mounts Google Drive, starts the FastAPI backend with Uvicorn, exposes it via ngrok, and prints the public URL.\n

In [ ]:
# 0) Optional: delete existing DB for a fresh migration run
# Run this cell first if you had partial migrations or "table already exists" errors.
import os
DB_PATHS = [
    "/content/drive/MyDrive/aktu_autonomy/aktu_autonomy.db",
    "/content/aktu_data/aktu_autonomy.db",
]
for path in DB_PATHS:
    try:
        if os.path.isfile(path):
            os.remove(path)
            print("Deleted (fresh DB):", path)
        else:
            print("No DB at:", path)
    except Exception as e:
        print("Skip", path, "(", e, ")")
print("Run the next cell to mount Drive, clone/pull, and run migrations.")

In [ ]:
# 1) Optional Drive mount, clone repo, install deps, run migrations

from google.colab import drive
import getpass
import os
import subprocess

# Try to mount Drive; if it fails (e.g. "credential propagation was unsuccessful"), use local storage
USE_DRIVE = True
try:
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Drive mount failed:", type(e).__name__, "- using local storage for this session.")
    USE_DRIVE = False

# --- Configure repo location and GitHub URL ---
REPO_DIR = "/content/aktu-autonomy-portal"
REPO_URL = os.environ.get("AKTU_REPO_URL", "https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application.git")

# --- Clone or pull ---
if not os.path.exists(REPO_DIR):
    print("Cloning repo…")
    token = getpass.getpass("GitHub PAT (repo scope): ")
    auth_url = REPO_URL.replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", auth_url, REPO_DIR], check=True)
else:
    print("Repo already exists; pulling latest changes…")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# --- Configure backend paths: Drive if mounted, else local ---
if USE_DRIVE:
    os.environ["AKTU_DB_PATH"] = "/content/drive/MyDrive/aktu_autonomy/aktu_autonomy.db"
    os.environ["UPLOAD_DIR"] = "/content/drive/MyDrive/aktu_autonomy/uploads"
else:
    os.environ["AKTU_DB_PATH"] = "/content/aktu_data/aktu_autonomy.db"
    os.environ["UPLOAD_DIR"] = "/content/aktu_data/uploads"
os.makedirs(os.path.dirname(os.environ["AKTU_DB_PATH"]), exist_ok=True)
os.makedirs(os.environ["UPLOAD_DIR"], exist_ok=True)

# --- Install Python deps ---
subprocess.run(["pip", "install", "-q", "-r", "backend/requirements.txt", "pyngrok"], check=True)

# --- Set env so app config loads (Alembic may load it) ---
os.environ.setdefault("JWT_SECRET", "colab-secret-do-not-use-in-production")
os.environ.setdefault("ENV", "dev")

# --- Run Alembic migrations (PYTHONPATH so subprocess finds app) ---
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
os.environ["PYTHONPATH"] = BACKEND_DIR
os.chdir(BACKEND_DIR)
result = subprocess.run(
    ["alembic", "upgrade", "head"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    err = (result.stderr or "") + (result.stdout or "")
    if "already exists" in err:
        print("Tables already exist (DB from a previous run). Stamping as up-to-date…")
        subprocess.run(["alembic", "stamp", "head"], check=True, cwd=BACKEND_DIR)
        print("Done. For a fresh database, delete the file at", os.environ.get("AKTU_DB_PATH"), "and re-run this cell.")
    else:
        print("Alembic stdout:", result.stdout)
        print("Alembic stderr:", result.stderr)
        result.check_returncode()
print("Backend dependencies installed and migrations applied.")


### Where to paste your ngrok authtoken

When you **run the next cell**, an **input box** will appear in the **output area directly under the cell** (it may say "Ngrok authtoken" or "Password").  

**Paste your ngrok authtoken into that box and press Enter.**  
(Get it from: https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# 2) Start Uvicorn + ngrok tunnel

import getpass
import os
import threading

import uvicorn
from pyngrok import ngrok

REPO_DIR = "/content/aktu-autonomy-portal"
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
os.chdir(BACKEND_DIR)

# Paste your ngrok authtoken in the input box that appears below when you run this cell, then press Enter.
ngrok_token = os.environ.get("NGROK_AUTHTOKEN") or getpass.getpass("Ngrok authtoken — paste in the box below and press Enter: ")
ngrok.set_auth_token(ngrok_token)

port = 8000
public_url = ngrok.connect(port, "http").public_url
print("Public URL:", public_url)
print("Health endpoint:", public_url + "/api/health")


def run_backend():
  # app.main:app, run inside backend directory
  uvicorn.run("app.main:app", host="0.0.0.0", port=port, reload=False)

thread = threading.Thread(target=run_backend, daemon=True)
thread.start()

print("Backend is starting… you can now call the health URL above.")


In [ ]:
# (Optional) Inspect environment / debug

import os
print("PWD:", os.getcwd())
print("AKTU_DB_PATH:", os.environ.get("AKTU_DB_PATH"))
print("UPLOAD_DIR:", os.environ.get("UPLOAD_DIR"))



## Testing and validation documentation

Open these docs (e.g. in a new tab) for testing and validation procedures, synthetic data, and reporting:

- [Testing and validation plan](https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application/blob/main/docs/TESTING_AND_VALIDATION_PLAN.md) — comprehensive plan (synthetic data, scenarios, report template).
- [Testing guide](https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application/blob/main/docs/TESTING_GUIDE.md) — how to run the app, seed data, manual and automated tests.
- [Testing report template](https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application/blob/main/docs/TESTING_REPORT_TEMPLATE.md) — process and product report template.